# Generate and inspect a PhaseGrid dataset
Run all cells for a smoke dataset using the grid-geometry environment. Generation refuses an existing destination; geometry uses procedural wallpaper.


## 1. Setup


In [ ]:
import json
import math
import os
import random
from collections import Counter
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import grid_geometry
from grid_geometry.dataset.generator import GENERATOR_REVISION, generate_dataset, _source_policy
from grid_geometry.dataset.sampling_config import GenerationConfig, SOURCE_CELL_SIZE
from grid_geometry.dataset.loader import PhaseDataset, inspect_dataset

DATA_DIR = Path(grid_geometry.__file__).resolve().parents[2] / "datasets"


## 2. Configuration
Edit the name, seed, workers or preset definitions as needed. To inspect an existing dataset, set its name, skip generation and continue from **Load saved dataset**. Customize `GenerationConfig` explicitly here only when a non-default policy is needed.


In [ ]:
PRESET = "smoke"
DATASET_NAME = PRESET
SEED = 42
WORKERS = os.process_cpu_count() or 1

PRESETS = {
    "smoke": {"train": 40, "val": 12, "test": 12},
    "development": {"train": 4000, "val": 500, "test": 500},
}
splits = ("train", "val", "test")
counts = PRESETS[PRESET]
destination = DATA_DIR / DATASET_NAME
config = GenerationConfig(**counts, seed=SEED, workers=WORKERS)
if GENERATOR_REVISION != 1:
    raise RuntimeError(f"Expected generator revision 1, found {GENERATOR_REVISION}")
print(f"Dataset destination: {destination}")
pprint({"preset": PRESET, "splits": counts,
        "generator_revision": GENERATOR_REVISION, "dataset_format": "phase-grid-v1",
        "seed": SEED, "available_cpus": os.process_cpu_count(), "workers": WORKERS})


## 3. Sampling policy overview
These are current production defaults; the saved configuration is authoritative when inspecting an existing run. Relative weights can be conditional.


In [ ]:
pprint({"source_policy": _source_policy(config), "native_cell_ranges": SOURCE_CELL_SIZE,
        "generation_settings": asdict(config)})


## 4. Generate dataset
Existing destinations, including incomplete runs, are never overwritten. Choose a different name for another run.


In [ ]:
if destination.exists():
    raise FileExistsError(f"Dataset destination already exists: {destination}")
generate_dataset(destination, config)
print(f"Generated dataset: {destination}")


## 5. Load saved dataset
Loading and the following sections are independent of generation in this session.


In [ ]:
for name in ("config.json", "manifest.jsonl", "statistics.json"):
    if not (destination / name).is_file():
        raise FileNotFoundError(destination / name)
metadata = json.loads((destination / "config.json").read_text())
statistics = json.loads((destination / "statistics.json").read_text())
with (destination / "manifest.jsonl").open() as manifest:
    records = [json.loads(line) for line in manifest if line.strip()]
print(f"Loaded dataset: {destination}")
pprint({key: metadata.get(key) for key in
        ("experiment", "generator", "generator_revision", "status", "seed", "workers", *splits)})


## 6. Verify dataset
Verification uses the saved configuration. The production inspector checks image/metadata pairing, counts, dimensions, targets, masks, manifest identities and cross-split board isolation; this can take time on a development dataset. Additional failures below raise errors, while missing optional diagnostics are reported as unavailable.


In [ ]:
for key, expected in {"experiment": "grid-geometry", "generator": "phase-grid-v1",
                      "generator_revision": 1, "status": "complete"}.items():
    if metadata.get(key) != expected:
        raise ValueError(f"Integrity failure: {key}: expected {expected!r}, found {metadata.get(key)!r}")
print("Production integrity inspection:")
pprint(inspect_dataset(destination))

def optional_check(label, value):
    print(f"{'UNAVAILABLE' if value is None else 'PASS' if value else 'FAIL'}: {label}")
    if value is False:
        raise ValueError(f"Integrity failure: {label}")

for record in records:
    split, name = record["split"], record["sample_id"]
    if split not in splits or Path(name).name != name:
        raise ValueError(f"Integrity failure: invalid manifest identity {split}/{name}")
    saved = json.loads((destination / split / f"{name}.json").read_text())
    if saved != record:
        raise ValueError(f"Integrity failure: manifest metadata differs for {split}/{name}")
for split in splits:
    actual = sum(r.get("split") == split for r in records)
    print(f"{split}: requested={metadata.get(split)}, manifest={actual}")
    accepted = statistics.get("accepted", {}).get("splits", {}).get(split, {}).get("totals", {}).get("images")
    optional_check(f"{split} accepted images", accepted == actual if accepted is not None else None)
    legacy = statistics.get("splits", {}).get(split, {})
    optional_check(f"{split} statistics samples", legacy["samples"] == actual if "samples" in legacy else None)
    attempt = statistics.get("attempts", {}).get("splits", {}).get(split)
    if attempt and all(k in attempt for k in ("attempted", "accepted", "rejected")):
        ok = attempt["attempted"] == attempt["accepted"] + attempt["rejected"] and attempt["accepted"] == actual
        if "retries" in legacy: ok = ok and sum(legacy["retries"].values()) == attempt["rejected"]
        optional_check(f"{split} source attempts", ok)
    else:
        optional_check(f"{split} source attempts", None)
accepted_all = statistics.get("accepted", {}).get("overall", {}).get("totals", {}).get("images")
optional_check("overall accepted images", accepted_all == len(records) if accepted_all is not None else None)
attempt_all = statistics.get("attempts", {}).get("overall")
if attempt_all and all(k in attempt_all for k in ("attempted", "accepted", "rejected")):
    optional_check("overall source attempts", attempt_all["attempted"] == attempt_all["accepted"] + attempt_all["rejected"] and attempt_all["accepted"] == len(records))
else:
    optional_check("overall source attempts", None)

positives = [r for r in records if r.get("board_id") is not None]
pitch_checked = category_checked = 0
for r in positives:
    requested, x, y, aspect = (r.get(k) for k in ("requested_pitch", "effective_pitch_x", "effective_pitch_y", "anisotropy_ratio"))
    if all(v is not None for v in (requested, x, y, aspect)):
        pitch_checked += 1
        if not (math.isclose(x, requested, rel_tol=1e-7, abs_tol=1e-7) and math.isclose(y, requested * aspect, rel_tol=1e-7, abs_tol=1e-7)):
            raise ValueError(f"Integrity failure: requested/realized pitch for {r['split']}/{r['sample_id']}")
    if r.get("requested_category") is not None and r.get("category") is not None:
        category_checked += 1
        if r["requested_category"] != r["category"]:
            raise ValueError(f"Integrity failure: requested/realized crop category for {r['split']}/{r['sample_id']}")
        crop, original = r.get("crop"), r.get("original_geometry")
        if crop and original and original.get("visible_box"):
            left, top, right, bottom = original["visible_box"]
            sides = sum((crop[0] < left < crop[2], crop[0] < right < crop[2],
                         crop[1] < top < crop[3], crop[1] < bottom < crop[3]))
            if sides != {"complete": 4, "interior": 0, "side": 1, "corner": 2}.get(r["category"]):
                raise ValueError(f"Integrity failure: crop geometry for {r['split']}/{r['sample_id']}")
print(f"PASS: requested/realized pitch for {pitch_checked} of {len(positives)} positive samples (y includes anisotropy)" if pitch_checked else "UNAVAILABLE: requested/realized pitch")
print(f"PASS: requested/realized crop category for {category_checked} of {len(positives)} positive samples" if category_checked else "UNAVAILABLE: requested/realized crop category")
budget = metadata.get("standalone_source_budget")
standalone = [r for r in records if r.get("scene_mode") == "game" and r.get("source_size")]
if budget is not None:
    violations = [r for r in standalone if max(r["source_size"]) > budget["max_dimension"] or r["source_size"][0] * r["source_size"][1] > budget["max_pixels"]]
    optional_check(f"standalone source budget ({len(standalone)} checked)", not violations if standalone else None)
else:
    print("UNAVAILABLE: standalone source budget is disabled or absent")
print("PASS: saved identity, completion, file pairing, targets, manifest and split isolation")


## 7. Analyze sampling distributions and performance
Weights, objective selections, source attempts, accepted images, source desktops, fill desktops and boards have distinct denominators. A source retry does not select a new learning objective. Missing data is excluded, not inferred.


In [ ]:
performance = statistics.get("performance", {})
environment = metadata.get("provenance", {}).get("environment", {})
print("Saved performance (seconds and images/second):")
for label, value in {
    "accepted images": performance.get("accepted_image_count"),
    "workers": metadata.get("workers"),
    "available CPUs": environment.get("available_cpu_count"),
    "logical CPUs": environment.get("logical_cpu_count"),
    "overall generation seconds": performance.get("overall_generation_seconds"),
    "sample generation/export seconds": performance.get("sample_generation_export_seconds"),
    "provenance collection seconds": performance.get("provenance_collection_seconds"),
    "final statistics/metadata preparation seconds": performance.get("final_statistics_metadata_preparation_seconds"),
    "overall images/second": performance.get("overall_images_per_second"),
    "sample generation images/second": performance.get("sample_generation_images_per_second"),
}.items():
    if value is None:
        displayed = "unavailable"
    elif "seconds" in label:
        displayed = f"{value:.3f}"
    elif "second" in label:
        displayed = f"{value:.2f}"
    else:
        displayed = str(value)
    print(f"  {label}: {displayed}")
print("  asset preparation: not applicable (procedural wallpaper)")
print("Overall includes setup between phases; final statistics and completed configuration writes are excluded.")


In [ ]:
def format_summary(value):
    """Round displayed floats while preserving counts and unavailable values."""
    if isinstance(value, float):
        return float(f"{value:.4g}")
    if isinstance(value, dict):
        return {k: format_summary(v) for k, v in value.items()}
    if isinstance(value, list):
        return [format_summary(v) for v in value]
    return value


def show_fitting(fitting):
    if not fitting:
        print("Fitting: unavailable")
        return
    print("Fitting by applicable boards (budget and placement have distinct denominators):")
    for group, buckets in fitting.items():
        for label, item in buckets.items():
            reduction = item.get("total_reduction", {})
            values = {key: item.get(key) for key in (
                "applicable_boards", "any_fitted", "any_fitted_rate",
                "budget_applicable_boards", "budget_fitted", "budget_fitting_rate",
                "placement_applicable_boards", "placement_fitted", "placement_fitting_rate")}
            values["total_reduction"] = {key: reduction.get(key) for key in ("count", "mean", "median", "p95", "max")}
            print(f"  {group}/{label}: {format_summary(values)}")

accepted = statistics.get("accepted", {}).get("overall", {})
attempts = statistics.get("attempts", {}).get("overall", {})
categorical = accepted.get("categorical", {})
numerical = accepted.get("numerical", {})
objectives = attempts.get("objectives", {}).get("by", {})
print("Selected saved settings:")
pprint(format_summary({k: metadata.get(k) for k in ("negative_fraction", "desktop_fraction", "standalone_source_budget", "context_weights", "padding_example_fraction")}))
print("Accepted totals:"); pprint(format_summary(accepted.get("totals", "unavailable")))

def distribution(title, values, unit):
    if not values:
        print(f"{title}: unavailable or no observations")
        return
    total = sum(values.values())
    print(f"\n{title} — {unit}, denominator={total}")
    for key, value in values.items():
        print(f"  {key}: {value} ({100 * value / total:.1f}%)" if total else f"  {key}: {value}")

def accepted_distribution(key, unit):
    distribution("Accepted " + key, categorical.get(key, {}).get("counts", {}), unit)

distribution("Configured positive categories", metadata.get("categories", {}), "relative weights, conditional on positive")
distribution("Selected categories", {k: v["selected"] for k, v in objectives.get("category", {}).items()}, "positive objectives")
accepted_distribution("realized_positive_category", "accepted positive images")
distribution("Configured pitch bins", {f"{b[0]:g}-{b[1]:g}": b[2] for b in metadata.get("pitch_bins", [])}, "relative weights, conditional on positive")
distribution("Selected pitch bins", {k: v["selected"] for k, v in objectives.get("pitch_bin", {}).items()}, "positive objectives")
accepted_distribution("requested_pitch_bin", "accepted positive images")
for axis in ("x", "y"):
    distribution("Realized pitch bin " + axis, Counter(str(r["actual_pitch_bin_" + axis]) for r in positives if r.get("actual_pitch_bin_" + axis) is not None), "accepted positives with bin available")
for key in ("category", "source_mode", "generated_fill", "context_type", "fill_procedural_subtype", "padding_side", "resampling_x", "resampling_y", "anisotropy_selected", "augmentation_resampling", "augmentation_blur"):
    accepted_distribution(key, "accepted images with field available")
for prefix in ("source_desktops_", "fill_desktops_"):
    for key in ("resolution_group", "display_scale", "shell", "style", "wallpaper_category", "wallpaper_subtype"):
        accepted_distribution(prefix + key, "applicable " + prefix.rstrip("_") + " observations")
for key in ("style", "theme", "requested_zoom", "requested_display_scale", "sampled_cell_size", "requested_cell_size", "resolved_cell_size"):
    accepted_distribution("board_" + key, "accepted boards with field available")
print("Source attempt buckets (attempt-weighted; availability shown separately):")
pprint(format_summary(attempts.get("by", "unavailable")))
print("Objective selections and their source retries:"); pprint(format_summary(attempts.get("objectives", "unavailable")))
print("Attempt availability:"); pprint(attempts.get("availability", "unavailable"))
print("Rejection reasons and retry frequency:")
for key in ("attempted", "accepted", "rejected", "rejection_reasons", "attempts_per_sample", "fitting_observations"):
    print(key, format_summary(attempts.get(key, "unavailable")))
print("Fitting markers are not rejection causes; unavailable joint distributions are not inferred.")
fills = [(r.get("fill_desktop") or {}).get("screen", {}).get("scale") for r in records if r.get("fill_desktop")]
print("Fill desktop scale 1.0:", all(v == 1.0 for v in fills) if fills else "unavailable", "n=", len(fills))


In [ ]:
metrics = ("board_pitch_x", "board_pitch_y", "source_width", "source_height", "source_pixels",
           "total_reduction", "budget_reduction", "placement_reduction", "desired_pitch",
           "final_pitch_x", "final_pitch_y", "pitch_error_x", "pitch_error_y", "anisotropy_ratio",
           "resampling_scale_x", "resampling_scale_y", "padding_extent", "known_fraction", "padding_fraction",
           "grid_bbox_image_fraction", "grid_coverage_fraction", "phase_supervision_fraction", "grid_occupancy_fraction")
print("Numerical summaries (count is the applicable image or board denominator):")
for key in metrics:
    entry = numerical.get(key)
    print(key, format_summary({k: entry.get(k) for k in ("count", "min", "median", "mean", "p05", "p95", "max")}) if entry else "unavailable")
show_fitting(accepted.get("fitting"))
print("Source budget:", metadata.get("standalone_source_budget"), "checked standalone sources:", len(standalone))
fig, axes = plt.subplots(2, 2, figsize=(12, 6), squeeze=False)
for ax, key in zip(axes.flat, ("final_pitch_x", "resampling_scale_x", "grid_coverage_fraction", "padding_fraction")):
    entry = numerical.get(key, {})
    hist = entry.get("histogram", {})
    edges, values = hist.get("edges", []), hist.get("counts", [])
    if values and len(edges) == len(values) + 1:
        ax.bar(range(len(values)), values)
        ax.set_xticks(range(len(values)), [f"{edges[i]:g}–{edges[i+1]:g}" for i in range(len(values))], rotation=45, ha="right", fontsize=7)
        ax.set_title(f"{key} (n={entry.get('count')}; under={hist.get('underflow')}, over={hist.get('overflow')})")
    else:
        ax.text(.5, .5, "Unavailable", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(key)
    ax.set_ylabel("observations")
fig.tight_layout()
plt.show()


## 8. Visualize dataset samples
Saved images and production target generation provide the eight panels. The yellow lattice is ideal; the cyan rectangle is the visible grid. Random selections are reproducible and repeated selections are combined. Absent categories are reported.


In [ ]:
rng = random.Random(SEED)
selected = {}
def choose(label, record):
    if record is None:
        print(f"Unavailable: {label}")
    else:
        key = (record["split"], record["sample_id"])
        selected.setdefault(key, {"record": record, "reasons": []})["reasons"].append(label)

def first(predicate):
    return next((r for r in records if predicate(r)), None)

def extreme(label, source, key, maximum=False):
    values = [r for r in source if key(r) is not None]
    choose(label, (max if maximum else min)(values, key=key) if values else None)

for r in rng.sample(records, min(2, len(records))): choose("random", r)
for category in ("complete", "interior", "side", "corner", "negative", "hard_negative"):
    choose(category, first(lambda r: r.get("category") == category))
extreme("small pitch", positives, lambda r: r.get("effective_pitch_x"))
extreme("large pitch", positives, lambda r: r.get("effective_pitch_x"), maximum=True)
extreme("low grid coverage", positives, lambda r: r.get("grid_coverage_fraction"))
for state, predicate in (("upsampling", lambda v: v > 1 + 1e-9), ("downsampling", lambda v: v < 1 - 1e-9)):
    choose(state, first(lambda r: any(predicate(v) for v in ((r.get("transform") or {}).get("scale_x"), (r.get("transform") or {}).get("scale_y")) if v is not None)))
for context in ("procedural", "desktop"):
    choose(context + " fill", first(lambda r: r.get("background_visible") and r.get("context_type") == context))
choose("intentional padding", first(lambda r: r.get("padding_type") is not None))
choose("source fitting", first(lambda r: r.get("sampled_cell_size") is not None and r.get("resolved_cell_size") is not None and r["sampled_cell_size"] > r["resolved_cell_size"]))
choose("standalone budget reduction", first(lambda r: r.get("scene_mode") == "game" and r.get("sampled_cell_size") is not None and r.get("requested_cell_size") is not None and r["sampled_cell_size"] > r["requested_cell_size"]))

datasets = {}
for item in selected.values():
    r = item["record"]
    split = r["split"]
    if split not in datasets: datasets[split] = PhaseDataset(destination, split)
    dataset = datasets[split]
    sample = dataset.load_sample(dataset.names.index(r["sample_id"]))
    targets = sample.targets.numpy()
    mask = sample.phase_mask.numpy()
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
    axes = axes.ravel()
    axes[0].imshow(sample.image.permute(1, 2, 0), extent=(0, dataset.size[0], dataset.size[1], 0))
    axes[0].set_title(", ".join(item["reasons"]) + " / RGB", fontsize=9)
    geometry = r.get("geometry")
    if geometry:
        w, h = dataset.size
        ox, oy, px, py = geometry["origin_x"], geometry["origin_y"], geometry["pitch_x"], geometry["pitch_y"]
        rows, columns = r["board_dimensions"]
        left, right = max(0, ox), min(w, ox + columns * px)
        top, bottom = max(0, oy), min(h, oy + rows * py)
        if left < right and top < bottom:
            for x in ox + np.arange(columns + 1) * px:
                if 0 <= x <= w: axes[0].plot([x, x], [top, bottom], color="yellow", lw=.5)
            for y in oy + np.arange(rows + 1) * py:
                if 0 <= y <= h: axes[0].plot([left, right], [y, y], color="yellow", lw=.5)
        vl, vt, vr, vb = geometry["visible_box"]
        axes[0].add_patch(Rectangle((vl, vt), vr-vl, vb-vt, fill=False, edgecolor="cyan", lw=1.2, linestyle="--"))
        axes[0].set_xlim(0, w); axes[0].set_ylim(h, 0)
    for i, title in enumerate(("X cosine", "X sine", "Y cosine", "Y sine")):
        axes[i+1].imshow(np.ma.masked_where(~mask, targets[i]), cmap="coolwarm", vmin=-1, vmax=1)
        axes[i+1].set_title(title)
    axes[5].imshow(targets[4], cmap="gray", vmin=0, vmax=1); axes[5].set_title("Soft occupancy")
    axes[6].imshow(mask, cmap="gray", vmin=0, vmax=1); axes[6].set_title("Phase supervision mask")
    axes[7].imshow(sample.occupancy_mask, cmap="gray", vmin=0, vmax=1); axes[7].set_title("Occupancy supervision mask")
    for ax in axes: ax.axis("off")
    pitch = (r.get("effective_pitch_x"), r.get("effective_pitch_y"))
    details = f" | pitch=({pitch[0]:.2f}, {pitch[1]:.2f})" if all(v is not None for v in pitch) else ""
    coverage = r.get("grid_coverage_fraction")
    padding = r.get("padding_fraction")
    if coverage is not None: details += f" | coverage={coverage:.1%}"
    if padding is not None: details += f" | padding={padding:.1%}"
    fig.suptitle(f"{split}/{r['sample_id']} | {r.get('category')} | {r.get('style')}{details}", fontsize=10)
    plt.show()
